# No-show Prediction

This notebook analyzes healthcare appointment data to predict patient no-shows. The workflow includes:

- Reading and loading the dataset
- Data cleaning and schema definition
- Handling missing values and dropping unnecessary columns
- Identifying and scaling numeric features for modeling

The goal is to prepare the data for building predictive models to identify factors influencing patient attendance.

In [0]:
%skip
%run ./setup/import_dataset
%run ./setup/data_preprocessing
%run ./setup/features_engineering

### Read in scaled features

In [0]:
train_scaled = spark.table("default.no_show_train_scaled_tbl")
test_scaled = spark.table("default.no_show_test_scaled_tbl")

In [0]:
print(f"Features read in from Delta Table")
print(f"  Train: {train_scaled.count():,} rows, {len(train_scaled.columns)} columns")
print(f"  Test: {test_scaled.count():,} rows, {len(test_scaled.columns)} columns")

In [0]:
display(train_scaled.limit(5))

#### LogisticRegression() call with regParam

In [0]:
%skip
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier

# Use scaled features
lr = LogisticRegression(
    # regParam=0.1,
    featuresCol="scaledFeatures",
    labelCol="Showed_up",
    maxIter=10,
    weightCol="weightCol"
)

In [0]:
import os
from pyspark.ml.classification import LogisticRegression

os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/no_show_volume"

# Re-instantiate LogisticRegression with correct featuresCol
lr = LogisticRegression(
    featuresCol="scaledFeatures",
    labelCol="Showed_up",
    maxIter=10,
    weightCol="weightCol"
)

# lr_cv = CrossValidator(
#     estimator=lr,
#     estimatorParamMaps=lr_param_grid,
#     evaluator=evaluator,
#     parallelism=6,  # Just a guess
# )

# # Call CrossValidator on the training data
# # cvModel = cv.fit(train_df)  # Fits a model to the input dataset with optional parameters.
# cvModel = lr_cv.fit(
#     train_scaled
# )  # Fits a model to the input dataset with optional parameters.
# cvModel.getNumFolds()  # Gets the value of numFolds or its default value.

#### Hyperparameter tuning

In [0]:
import numpy as np
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator, CrossValidatorModel

lr_param_grid = (ParamGridBuilder()
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    # .addGrid(lr.regParam, [np.logspace(0,4,20)])
    # .addGrid(lr.regParam, [1,3,5,7,9])
    .addGrid(lr.regParam, [1,3,9])
    # .addGrid(lr.solver, ['lbfgs','newton-cg','liblinear','sag','saga'])
    .addGrid(lr.maxIter, [100, 1000, 2500])
    # .addGrid(lr.maxIter, [100, 1000, 2500, 5000])
    .build()
) # End lr_param_grid

In [0]:
# Scorer
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol = "Showed_up",
    metricName = "areaUnderROC"
)
     

In [0]:
lr_cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=lr_param_grid,
    evaluator=evaluator,
    parallelism=6,  # Just a guess
)

# Call CrossValidator on the training data
# cvModel = cv.fit(train_df)  # Fits a model to the input dataset with optional parameters.
cvModel = lr_cv.fit(
    train_scaled
)  # Fits a model to the input dataset with optional parameters.
cvModel.getNumFolds()  # Gets the value of numFolds or its default value.

In [0]:
# Scorer
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol = "Showed_up",
    metricName = "areaUnderROC"
)

### CrossValidator

In [0]:
%skip
import os
from pyspark.ml.classification import LogisticRegression

os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/no_show_volume"

# Re-instantiate LogisticRegression with correct featuresCol
lr = LogisticRegression(
    featuresCol="scaledFeatures",
    labelCol="Showed_up",
    maxIter=10,
    weightCol="weightCol"
)

lr_cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=lr_param_grid,
    evaluator=evaluator,
    parallelism=6,  # Just a guess
)

# Call CrossValidator on the training data
# cvModel = cv.fit(train_df)  # Fits a model to the input dataset with optional parameters.
cvModel = lr_cv.fit(
    train_scaled
)  # Fits a model to the input dataset with optional parameters.
cvModel.getNumFolds()  # Gets the value of numFolds or its default value.

In [0]:
# pipeline = Pipeline(stages=[indexer, encoder, vector_assembler])
stages=[indexer, encoder, vector_assembler, lr]
pipeline = Pipeline().setStages(stages)

In [0]:
%skip
# train_scaled already has indexing/encoding applied, fit LR directly
from pyspark.ml.classification import LogisticRegression

lr_model = lr.fit(train_scaled)
lr_predictions = lr_model.transform(test_scaled)

In [0]:
cvModel = lr_cv.fit(train_scaled) # use CrossValidator
lr_model = cvModel.bestModel
lr_predictions = lr_model.transform(test_scaled)

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql import functions as F

In [0]:
test_encoded = lr_model.transform(test_df)

lr_test_data = test_encoded.select(
    "features",
    F.col("Showed_up").alias("label"))

#### Log LR in MLFlow run

In [0]:
import mlflow       # Experiment tracking & Record ML runs
import mlflow.spark # Log model artifacts
import os
from mlflow.models import infer_signature

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/no_show_volume"

# EXPERIMENT_NAME = "/Users/asanders4205@gmail.com/predict_no_show"
# TARGET = "Showed_up"

In [0]:
#  columns: prediction, label, weight (optional) and probabilityCol (only for logLoss)
multi_evaluator = MulticlassClassificationEvaluator(
    predictionCol = 'prediction',
    labelCol = 'Showed_up'
)

#### Evaluation of LR

In [0]:
evaluator = BinaryClassificationEvaluator(
  rawPredictionCol = 'rawPrediction',
  labelCol = 'Showed_up',
  metricName = 'areaUnderROC')
  #   weightCol = 'weightCol')
  
auc = evaluator.evaluate(lr_predictions)

In [0]:
# print(multi_evaluator.explainParams())
recall = multi_evaluator.evaluate(lr_predictions, {multi_evaluator.metricName: "weightedRecall"})
f1 = multi_evaluator.evaluate(lr_predictions, {multi_evaluator.metricName: "f1"})
accuracy = multi_evaluator.evaluate(lr_predictions, {multi_evaluator.metricName: "accuracy"})
precision = multi_evaluator.evaluate(lr_predictions, {multi_evaluator.metricName: "weightedPrecision"})

In [0]:
signature_lr = infer_signature(train_df, lr_predictions)

In [0]:
with mlflow.start_run(run_name="LogisticRegressionModel") as run:
    mlflow.spark.log_model(lr_model, "LogisticRegressionModel") # skipped — Serverless 256MB limit
    mlflow.log_param("maxIter", 10)
    mlflow.log_param("featuresCol", "features")
    mlflow.log_param("labelCol", "Showed_up")
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)

In [0]:
# Inspect the LoggedModel, now with metrics
logged_model = mlflow.get_logged_model(model_info.model_id)
print(logged_model.model_id, logged_model.metrics)